# Train the TransitEye candidate classifier
Open this notebook in Google Colab, select Runtime → Change runtime type → GPU (CPU also works), then Run all. Upload **matched_candidate_sequences.npz** from your project workspace under `.transiteye-workspace/models/labeled_candidate_sequence_v1/` when prompted. Download the results ZIP before disconnecting. No Drive mount or repository clone is required.

## 1. Load the frozen development data
The extraction run found matching candidates for 3,940 of 6,255 development sources. The remaining 2,315 had no candidate match. Each input has 256 phase bins for delivered flux, detrended flux and bin occupancy. The model distinguishes transit-like and eclipse-like catalog proxies.

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import hashlib, json, logging, math, platform, random, shutil, tempfile
from pathlib import Path
from collections import Counter
from typing import Any
import joblib, numpy as np, scipy, sklearn, torch
from scipy.optimize import minimize_scalar
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from torch import Tensor, nn
from torch.utils.data import DataLoader, TensorDataset

from google.colab import files
filename = 'matched_candidate_sequences.npz'
if not Path(filename).is_file():
    files.upload()
path = Path(filename)
expected_hash = '9c1323f0470d8e541603fd55a48644b5e0fd61d74d3372635a4c57da3a670130'
if hashlib.sha256(path.read_bytes()).hexdigest() != expected_hash:
    raise ValueError('Upload the development sequence file specified above.')
with np.load(path, allow_pickle=False) as data:
    seq = data['sequence']
    groups, labels, splits = (data[key].astype(str) for key in ('source_group_id','label','split'))
CLASSES = ('transit_like', 'eclipse_like')
PARTITIONS = ('train', 'calibration', 'selection')
SEED = 42
if (seq.shape != (3940, 3, 256) or not np.isfinite(seq).all()
    or not (len(groups) == len(set(groups)) == len(labels) == len(splits) == len(seq))
    or set(labels) != set(CLASSES) or set(splits) != set(PARTITIONS)):
    raise ValueError('Unexpected sequence data or partitions')
records = [dict(source_group_id=g, label=y, split=s, sequence=x)
           for g,y,s,x in zip(groups,labels,splits,seq)]
print(Counter(zip(splits, labels)))

## 2. Define the models
These cells capture the project’s candidate CNN and training functions so the notebook runs on its own. The random forest uses the same 768 flux and occupancy values as the CNN. Both fit on the training partition. Selection macro-F1 chooses the CNN epoch, then the calibration partition sets the temperature.

In [ ]:
class _ResidualBlock(nn.Module):
    """Small one-dimensional residual block with an optional downsampling path."""

    def __init__(self, input_channels: int, output_channels: int, stride: int, kernel_size: int):
        super().__init__()
        padding = kernel_size // 2
        self.main = nn.Sequential(
            nn.Conv1d(
                input_channels,
                output_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                bias=False,
            ),
            nn.GroupNorm(8, output_channels),
            nn.GELU(),
            nn.Conv1d(
                output_channels,
                output_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            nn.GroupNorm(8, output_channels),
        )
        self.skip = (
            nn.Identity()
            if input_channels == output_channels and stride == 1
            else nn.Sequential(
                nn.Conv1d(input_channels, output_channels, kernel_size=1, stride=stride, bias=False),
                nn.GroupNorm(8, output_channels),
            )
        )
        self.activation = nn.GELU()

    def forward(self, values: Tensor) -> Tensor:
        return self.activation(self.main(values) + self.skip(values))

class CandidateSequenceCNN(nn.Module):
    """Classify fixed-width sequences centered on a detected candidate event.

    Inputs have shape ``(batch, 3, sequence_length)`` with channels for delivered
    normalized flux, detrended flux, and bin occupancy. Period, epoch, duration,
    target identity, catalog fields, and dataset split are not model inputs.
    """

    def __init__(self, class_count: int = 4, dropout: float = 0.25) -> None:
        super().__init__()
        if class_count < 2:
            raise ValueError("class_count must be at least two.")
        if not 0.0 <= dropout < 1.0:
            raise ValueError("dropout must be in [0, 1).")
        self.input_channels = 3
        self.encoder = nn.Sequential(
            nn.Conv1d(3, 32, kernel_size=9, padding=4, bias=False),
            nn.GroupNorm(8, 32),
            nn.GELU(),
            _ResidualBlock(32, 32, stride=1, kernel_size=7),
            _ResidualBlock(32, 64, stride=2, kernel_size=7),
            _ResidualBlock(64, 128, stride=2, kernel_size=5),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * 8, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, class_count),
        )

    def forward(self, values: Tensor) -> Tensor:
        """Return uncalibrated class logits for a batch of candidate sequences."""
        if values.ndim != 3 or values.shape[1] != self.input_channels:
            raise ValueError("Expected input shaped (batch, 3, sequence_length).")
        encoded = self.encoder(values)
        if encoded.shape[-1] % 8 != 0:
            raise ValueError("Encoded sequence length must be divisible into eight segments.")
        pooled_segments = encoded.reshape(
            encoded.shape[0], encoded.shape[1], 8, encoded.shape[-1] // 8
        ).mean(dim=-1).flatten(start_dim=1)
        return self.classifier(pooled_segments)

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def _temperature(logits: Tensor, labels: Tensor) -> float:
    logits = logits.detach().cpu()
    labels = labels.detach().cpu()

    def objective(log_t: float) -> float:
        return float(nn.functional.cross_entropy(logits / math.exp(log_t), labels).item())

    result = minimize_scalar(objective, bounds=(-3.0, 3.0), method="bounded")
    if not result.success or not math.isfinite(float(result.fun)):
        raise RuntimeError("Temperature scaling failed on the calibration partition")
    return float(math.exp(result.x))

def _grouped_scores(labels: list[int], logits: np.ndarray) -> dict[str, Any]:
    predictions = logits.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "confusion_matrix": confusion_matrix(labels, predictions, labels=list(range(len(CLASSES)))).tolist(),
        "class_order": list(CLASSES),
        "sample_count": len(labels),
    }

def _fit_models(
    records: list[dict[str, Any]],
    output_dir: Path,
    *,
    maximum_epochs: int = 100,
    batch_size: int = 32,
    device: str = "cuda",
) -> dict[str, Any]:
    class_index = {label: index for index, label in enumerate(CLASSES)}
    by_split = {name: [row for row in records if row["split"] == name] for name in PARTITIONS}
    counts = {name: Counter(row["label"] for row in rows) for name, rows in by_split.items()}
    if any(not rows for rows in by_split.values()):
        raise RuntimeError(f"A development partition has no matched candidates: {counts}")
    if any(any(counts[name].get(label, 0) == 0 for label in CLASSES) for name in PARTITIONS):
        raise RuntimeError(f"Both labels must have matched candidates in all development partitions: {counts}")

    x = {
        name: np.stack([np.asarray(row["sequence"], dtype=np.float32) for row in rows])
        for name, rows in by_split.items()
    }
    y = {
        name: np.asarray([class_index[row["label"]] for row in rows], dtype=np.int64)
        for name, rows in by_split.items()
    }
    groups = {name: [row["source_group_id"] for row in rows] for name, rows in by_split.items()}
    if any(len(group_list) != len(set(group_list)) for group_list in groups.values()):
        raise RuntimeError("Candidate set must contain at most one observation per TIC")
    if any(set(groups[left]) & set(groups[right]) for i, left in enumerate(PARTITIONS) for right in PARTITIONS[i + 1:]):
        raise RuntimeError("TIC source group leaked between development partitions")

    # Frozen same-candidate tabular baseline: random forest over the same 3x256 sequence bins.
    x_rf = {name: values.reshape(len(values), -1) for name, values in x.items()}
    baseline = RandomForestClassifier(
        n_estimators=300,
        max_depth=14,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=1,
    )
    baseline.fit(x_rf["train"], y["train"])
    baseline_selection = _grouped_scores(y["selection"].tolist(), baseline.predict_proba(x_rf["selection"]))
    baseline_path = output_dir / "same_candidate_random_forest.joblib"
    joblib.dump(baseline, baseline_path)

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    model = CandidateSequenceCNN(class_count=len(CLASSES)).to(device)
    y_train = torch.tensor(y["train"], dtype=torch.long)
    x_train = torch.from_numpy(x["train"])
    x_cal = torch.from_numpy(x["calibration"])
    y_cal = torch.tensor(y["calibration"], dtype=torch.long)
    x_sel = torch.from_numpy(x["selection"]).to(device)
    train_support = torch.bincount(y_train, minlength=len(CLASSES)).float()
    class_weights = torch.sqrt(train_support.sum() / train_support.clamp_min(1.0))
    class_weights /= class_weights.mean()
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True, generator=generator)

    best_score = -1.0
    best_epoch = 0
    best_state: dict[str, Tensor] | None = None
    stale = 0
    history: list[dict[str, float]] = []
    for epoch in range(1, maximum_epochs + 1):
        model.train()
        for batch_x, batch_y in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_x.to(device)), batch_y.to(device))
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            logits = model(x_sel).cpu().numpy()
        score = float(f1_score(y["selection"], logits.argmax(axis=1), average="macro", zero_division=0))
        history.append({"epoch": float(epoch), "selection_macro_f1": score})
        LOGGER.info("epoch %d/%d selection macro-F1=%.4f", epoch, maximum_epochs, score)
        if score > best_score:
            best_score, best_epoch = score, epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= 15:
                break
    if best_state is None:
        raise RuntimeError("CNN produced no selectable checkpoint")
    model.load_state_dict(best_state)
    model.to(device).eval()
    with torch.no_grad():
        calibration_logits = model(x_cal.to(device)).cpu()
        selection_logits = model(x_sel).cpu()
    temperature = _temperature(calibration_logits, y_cal)
    calibrated_selection_logits = selection_logits.numpy() / temperature

    state_path = output_dir / "sequence_model_state.pt"
    torch.save(best_state, state_path)
    report = {
        "model_id": "labeled_candidate_sequence_cnn_binary_v1",
        "training_source": "fresh_initialization_no_pretrained_weights",
        "class_order": list(CLASSES),
        "partition_candidate_counts": {name: len(rows) for name, rows in by_split.items()},
        "partition_class_counts": {name: dict(counts[name]) for name in PARTITIONS},
        "baseline": {
            "model": "RandomForestClassifier on the same 768 sequence-bin features",
            "selection": baseline_selection,
            "artifact_sha256": sha256_file(baseline_path),
        },
        "cnn": {
            "architecture": "CandidateSequenceCNN(class_count=2)",
            "best_epoch_selected_on_selection": best_epoch,
            "selection_macro_f1_at_selected_epoch": best_score,
            "selection_after_temperature_scaling": _grouped_scores(y["selection"].tolist(), calibrated_selection_logits),
            "calibration_temperature": temperature,
            "artifact_sha256": sha256_file(state_path),
        },
        "training_history": history,
        "torch_version": torch.__version__,
        "torch_cuda_version": torch.version.cuda,
        "device": device,
        "seed": SEED,
        "test_partitions_opened": False,
        "fresh_holdout_opened": False,
        "independent_performance_claim": False,
        "limitations": [
            "two-class research model only; stellar_variability and other_artifact are untrained",
            "catalog labels and candidate-event matches are proxies, not independent morphology adjudication",
            "selection metrics are for model selection and are not independent performance estimates",
            "unlabeled products and unmatched candidates were excluded, never treated as negatives",
            "candidate detection failures remain in the full extraction accounting",
            "the fresh holdout is sealed and was not used for fitting or selection",
        ],
    }
    return report

## 3. Train and inspect selection scores
A full run allows 100 epochs with early stopping. For a wiring check, set MAX_EPOCHS to 1. The confusion matrices use true classes on rows and predicted classes on columns. Selection scores describe matched candidates only. The existing holdout has already been evaluated.

In [ ]:
MAX_EPOCHS = 100
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_num_threads(2)
LOGGER = logging.getLogger('transiteye.colab')
logging.basicConfig(level=logging.INFO)
out = Path(tempfile.mkdtemp(prefix='transiteye-training-'))
print('Training on', device, flush=True)
report = _fit_models(records, out, maximum_epochs=MAX_EPOCHS, batch_size=32, device=device)
report.update(input_sha256=expected_hash, maximum_epochs=MAX_EPOCHS, batch_size=32,
              python_version=platform.python_version(), numpy_version=np.__version__,
              sklearn_version=sklearn.__version__, scipy_version=scipy.__version__,
              extraction_counts={'matched':3940,'no_candidate_match':2315})
report['limitations'][-1] = 'This training run uses development data only. The historical holdout is spent.'
(out/'training_report.json').write_text(json.dumps(report, indent=2))
print(json.dumps({'forest':report['baseline']['selection'], 'cnn':report['cnn']}, indent=2))

## 4. Save the run
The ZIP contains the random forest, CNN weights and report with class order, calibration temperature, input checksum and package versions. Keep this notebook with the ZIP for the architecture definition.

In [ ]:
files.download(shutil.make_archive(str(out), 'zip', out))